<a href="https://colab.research.google.com/github/pomellonn/strikt/blob/main/pairs_labeled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

постановка задачи: contrastive learning для переобучения эмбеддера для генерации кандидатов перед дедубликацией

на этой неделе:
1. разметка датасета на классы
duplicate (поведение водителя — поведение водителей, рост цен — повышение стоимости),
similar_not_duplicate (бпла — атака бпла, кофе — рост цен на кофе),
not_duplicate (поведение мужчин — поведение женщин, кошки — собаки).
то есть будет 3 колонки: объект_1, объект_2, тип связи
2. объединяем датасет
3. делаем сплит на train\test\val. подберем порог схожести и через StratifiedGroupKFold разделим на выборки.

### Установка


### Установка зависимостей
Установка библиотек для работы с эмбеддингами, лемматизацией и обработкой данных.

In [1]:
!pip install -q sentence-transformers pymorphy3 pymorphy3-dicts-ru


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 62.8 MB/s eta 0:00:00


### Импорт библиотек
Подключение основных модулей для обработки данных и подготовки датасета.

In [2]:
import csv, re
import pandas as pd
import numpy as np
from itertools import combinations
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

rows = []
with open('objects3.csv', encoding='utf-8') as f:
    r = csv.reader(f)
    next(r)  # header
    for row in r:
        rows.append(','.join(row).strip())

df = pd.DataFrame({'object_name': rows})
df = df[df['object_name'] != ''].drop_duplicates().reset_index(drop=True)
print(f'Объектов: {len(df)}')


Объектов: 4167


### Нормализация текста
Функция приводит тексты к лемматизированному представлению для поиска похожих объектов.

In [5]:
df['object_name'] = (
    df['object_name']
    .astype(str)
    .str.strip()
    .str.lower()
)

def lemma_sig(text):
    tokens = re.findall(r'\w+', text)
    lemmas = sorted(morph.parse(t)[0].normal_form for t in tokens)
    return ' '.join(lemmas)

df['lemma'] = df['object_name'].apply(lemma_sig)

groups = df.groupby('lemma')['object_name'].apply(list)

auto_dup_rows = []

for names in groups[groups.apply(len) > 1]:
    for a, b in combinations(names, 2):
        auto_dup_rows.append({
            'object_1': a,
            'object_2': b,
            'connection': 'duplicate'
        })

auto_dup_df = pd.DataFrame(auto_dup_rows)

print(f'Авто-duplicate по лемме: {len(auto_dup_df)} пар')

auto_dup_df.to_csv('auto_duplicates.csv', index=False)

unique_df = df.drop_duplicates(subset='lemma').reset_index(drop=True)

print(f'Уникальных по лемме объектов: {len(unique_df)}')

Авто-duplicate по лемме: 126 пар
Уникальных по лемме объектов: 4053


### Загрузка модели эмбеддингов
Инициализация модели SentenceTransformer и вычисление векторных представлений.

In [7]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
emb = model.encode(
    unique_df['object_name'].tolist(),
    batch_size=128,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

### Поиск кандидатов
Расчёт матрицы сходства и формирование пар для последующей разметки.

In [8]:
sim = util.cos_sim(emb, emb).cpu().numpy()
n = len(unique_df)
iu = np.triu_indices(n, k=1)

pairs_df = pd.DataFrame({
    'object_1': unique_df['object_name'].values[iu[0]],
    'object_2': unique_df['object_name'].values[iu[1]],
    'similarity': sim[iu]
}).sort_values('similarity', ascending=False).reset_index(drop=True)

pairs_df['connection'] = ''
print(f'Всего пар: {len(pairs_df)}')


Всего пар: 8211378


### Отбор сложных примеров
Выбор диапазона пар, наиболее полезных для ручной разметки.

In [9]:
TOP_N = 3000
to_label = pairs_df.head(TOP_N)
to_label.to_csv('pairs_to_label.csv', index=False)

In [10]:
to_label2 = pairs_df[3000:4000]
to_label2.to_csv('pairs_to_label2.csv', index=False)

Разметка была проведена с помощью claude code и ручной проверки

---
## Объединение датасета


Объединение автоматически и вручную размеченных данных.

In [11]:
df1 = pd.read_csv('auto_duplicates.csv', sep=';')
df2 = pd.read_csv('pairs_labeled.csv')
df3 = pd.read_csv('pairs_labeled2.csv', sep=';')
full = pd.concat([df1, df2, df3], ignore_index=True)
full = full.drop_duplicates(subset=['object_1', 'object_2'])
print(full['connection'].value_counts())
full.to_csv('objects3_pairs_labeled.csv', index=False)

connection
not_duplicate            2575
duplicate                 828
similar_not_duplicate     208
Name: count, dtype: int64


## сплит

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv("objects3_pairs_labeled.csv")

df = df.dropna(subset=["object_1", "object_2", "connection"]).copy()

df["object_1"] = df["object_1"].astype(str).str.strip()
df["object_2"] = df["object_2"].astype(str).str.strip()
df["connection"] = df["connection"].astype(str).str.strip()

print(df.shape)
print(df["connection"].value_counts())

(3611, 5)
connection
not_duplicate            2575
duplicate                 828
similar_not_duplicate     208
Name: count, dtype: int64


### Построение графа связей
Создание графа и поиск связанных компонентов.

In [13]:
import networkx as nx

G = nx.Graph()

for _, row in df.iterrows():
    G.add_edge(row["object_1"], row["object_2"])

components = list(nx.connected_components(G))

print("Количество компонент:", len(components))
print("Размер крупнейших компонент:",
      sorted([len(c) for c in components], reverse=True)[:20])

Количество компонент: 416
Размер крупнейших компонент: [231, 44, 42, 41, 37, 30, 27, 19, 17, 16, 13, 13, 12, 12, 11, 11, 11, 11, 11, 10]


### Назначение групп
Присвоение идентификаторов групп объектам.

In [14]:
object_to_group = {}

for group_id, component in enumerate(components):
    for obj in component:
        object_to_group[obj] = group_id

df["group"] = df["object_1"].map(object_to_group)

assert df["group"].notna().all()

print(df["group"].nunique(), "groups")

416 groups


### Проверка целостности
Контроль корректности сформированных групп.

In [15]:
assert (
    df["object_2"].map(object_to_group).values
    == df["group"].values
).all()

print("Group consistency: OK")

Group consistency: OK


### Подготовка сплитов
Разделение данных с учётом групп объектов.

In [16]:
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

X = df
y = df["connection"]
groups = df["group"]

folds = list(
    sgkf.split(X, y, groups=groups)
)

for i, (_, val_idx) in enumerate(folds):
    print(
        f"Fold {i}:",
        len(val_idx),
        df.iloc[val_idx]["connection"].value_counts(normalize=True).round(3).to_dict()
    )

Fold 0: 6 {'not_duplicate': 0.667, 'duplicate': 0.333}
Fold 1: 738 {'not_duplicate': 0.591, 'duplicate': 0.348, 'similar_not_duplicate': 0.061}
Fold 2: 627 {'not_duplicate': 0.587, 'duplicate': 0.297, 'similar_not_duplicate': 0.116}
Fold 3: 255 {'not_duplicate': 0.541, 'duplicate': 0.376, 'similar_not_duplicate': 0.082}
Fold 4: 1985 {'not_duplicate': 0.821, 'duplicate': 0.145, 'similar_not_duplicate': 0.035}


### Формирование train/val/test
Создание финальных выборок для обучения и оценки.

In [17]:
train_val_idx, test_idx = folds[0]

train_val = df.iloc[train_val_idx].copy()
test = df.iloc[test_idx].copy()

print("Train+Val:", len(train_val))
print("Test:", len(test))

Train+Val: 3605
Test: 6


In [18]:
sgkf_inner = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

inner_folds = list(
    sgkf_inner.split(
        train_val,
        train_val["connection"],
        groups=train_val["group"]
    )
)

train_idx, val_idx = inner_folds[0]

train = train_val.iloc[train_idx].copy()
val = train_val.iloc[val_idx].copy()

print("TRAIN:", len(train))
print("VAL:", len(val))
print("TEST:", len(test))

TRAIN: 3600
VAL: 5
TEST: 6


проыерка на пересечение пар

In [19]:
train_groups = set(train["group"])
val_groups = set(val["group"])
test_groups = set(test["group"])

print("TRAIN ∩ VAL:", len(train_groups & val_groups))
print("TRAIN ∩ TEST:", len(train_groups & test_groups))
print("VAL ∩ TEST:", len(val_groups & test_groups))

TRAIN ∩ VAL: 0
TRAIN ∩ TEST: 0
VAL ∩ TEST: 0


сохранение выборок в файлы

In [20]:
train_save = train.drop(columns=["group", "similarity"], errors="ignore")
val_save = val.drop(columns=["group", "similarity"], errors="ignore")
test_save = test.drop(columns=["group", "similarity"], errors="ignore")

train_save.to_csv("train.csv", index=False)
val_save.to_csv("val.csv", index=False)
test_save.to_csv("test.csv", index=False)

In [21]:
a = pd.read_csv('test.csv')
a.head()

,"object_1,object_2,connection",object_1,object_2,connection
0,NaN,жизненный уровень,уровень жизни,duplicate
1,NaN,личная жизнь автора,жизненный подход автора,not_duplicate
2,NaN,опыт автора,жизненный подход автора,not_duplicate
3,NaN,личная жизнь автора,опыт автора,not_duplicate
4,NaN,система власти РФ,политическая система РФ,duplicate
